# Ngày 8 — Structured output: trích xuất hóa đơn → JSON

Trích các trường có cấu trúc từ văn bản tự do bằng OpenAI (JSON mode + pydantic), có **retry** khi JSON hỏng, đo **độ chính xác** so với ground truth.

`pip install openai pydantic` · cần `OPENAI_API_KEY` trong env.

In [ ]:
import os
import json
import pandas as pd
from pydantic import BaseModel, ValidationError
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Chưa đặt OPENAI_API_KEY trong env!"
client = OpenAI()
MODEL = "gpt-4o-mini"

## Schema mong muốn

In [ ]:
class Invoice(BaseModel):
    vendor: str
    date: str          # yyyy-mm-dd
    total: float
    currency: str      # VD USD, VND

SCHEMA_HINT = 'Trả JSON đúng dạng: {"vendor": str, "date": "yyyy-mm-dd", "total": number, "currency": str}'

## 15 tài liệu mẫu + ground truth (có 1 tài liệu "khó" thiếu trường)

In [ ]:
docs = [
    "Invoice from Acme Corp dated 2026-01-05. Total amount due: $1,250.00 USD.",
    "Hóa đơn công ty Bến Thành, ngày 12/02/2026, tổng cộng 3.400.000 VND.",
    "Receipt - GlobalTech - 2026-03-01 - Amount: EUR 780.50",
    "Thanh toán cho Nhà cung cấp Sao Mai ngày 2026-03-15, số tiền 990000 VND.",
    "Bill: Northwind Traders. Date 2026-02-20. Grand total 542.30 USD.",
    "Hóa đơn Viettel, 2026-01-31, cước dịch vụ: 220,000 VND",
    "Invoice #A19 from Umbrella Inc, 2026-04-02, total 4300 USD",
    "Cửa hàng Minh Anh - 05/04/2026 - Tổng: 1.150.000 đồng",
    "Payment to Stark Industries on 2026-02-14 for USD 9,999.99",
    "Hóa đơn điện lực EVN tháng 3/2026, tổng tiền 875500 VND",
    "Invoice: Wayne Enterprises | 2026-03-28 | Total EUR 2,000",
    "Biên nhận công ty Hòa Bình, ngày 2026-04-10, 640000 VND",
    "From Initech, dated 2026-01-18, amount due 315.75 USD",
    "Hóa đơn Grab, 2026-04-11, 87,000 VND",
    "Thư cảm ơn quý khách đã mua hàng tại Cửa hàng ABC. Hẹn gặp lại!",  # <- KHÓ: không có ngày/tổng tiền
]

# ground truth (None = không có/không áp dụng). Tài liệu cuối cố tình thiếu.
ground_truth = [
    {"vendor": "Acme Corp", "total": 1250.0, "currency": "USD"},
    {"vendor": "Bến Thành", "total": 3400000, "currency": "VND"},
    {"vendor": "GlobalTech", "total": 780.5, "currency": "EUR"},
    {"vendor": "Sao Mai", "total": 990000, "currency": "VND"},
    {"vendor": "Northwind Traders", "total": 542.3, "currency": "USD"},
    {"vendor": "Viettel", "total": 220000, "currency": "VND"},
    {"vendor": "Umbrella Inc", "total": 4300, "currency": "USD"},
    {"vendor": "Minh Anh", "total": 1150000, "currency": "VND"},
    {"vendor": "Stark Industries", "total": 9999.99, "currency": "USD"},
    {"vendor": "EVN", "total": 875500, "currency": "VND"},
    {"vendor": "Wayne Enterprises", "total": 2000, "currency": "EUR"},
    {"vendor": "Hòa Bình", "total": 640000, "currency": "VND"},
    {"vendor": "Initech", "total": 315.75, "currency": "USD"},
    {"vendor": "Grab", "total": 87000, "currency": "VND"},
    None,  # tài liệu khó — không phải hóa đơn
]
print(len(docs), "tài liệu")

## Hàm trích xuất — JSON mode + parse an toàn + retry 1 lần

In [ ]:
SYSTEM = f"Bạn trích thông tin hóa đơn. Nếu văn bản KHÔNG phải hóa đơn hoặc thiếu trường, để giá trị null. {SCHEMA_HINT}"

def extract(doc, retries=1):
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": doc},
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            raw = resp.choices[0].message.content
            data = json.loads(raw)              # có thể ném JSONDecodeError
            return Invoice(**data).model_dump()  # validate schema; ném ValidationError nếu sai
        except (json.JSONDecodeError, ValidationError) as e:
            if attempt < retries:
                continue  # thử lại 1 lần
            return {"_error": f"{type(e).__name__}", "_need_review": True}
        except Exception as e:
            return {"_error": f"{type(e).__name__}: {e}", "_need_review": True}

extract(docs[0])

## Chạy trên toàn bộ 15 tài liệu

In [ ]:
results = [extract(d) for d in docs]
df = pd.DataFrame(results)
df.insert(0, "doc", [d[:45] + "..." for d in docs])
df

## Đo độ chính xác (trên các tài liệu có ground truth)

In [ ]:
correct = total = 0
for res, gt in zip(results, ground_truth):
    if gt is None:
        continue  # bỏ qua tài liệu khó khi tính accuracy trường
    total += 1
    # TODO: có thể so khớp lỏng hơn cho vendor (chứa chuỗi) — hiện so total & currency
    if res.get("total") == gt["total"] and res.get("currency") == gt["currency"]:
        correct += 1

print(f"Độ chính xác (total + currency): {correct}/{total} = {correct/total:.0%}")
print("Tài liệu khó được đánh dấu need_review:", results[-1])

## Nhận xét

_(Trường nào hay sai? Tài liệu 'khó' được xử lý ra sao — có crash không? Cách cải thiện prompt/schema?)_